# STAR-RIS RSMA - scalability theo N (V4 shard workflow)

Voi moi N trong n_sweep, MADDPG va TD3 duoc train lai tu dau. Quy trinh:

1. Cell 2 dung `main.load_config()` de resolve seed split (danh sach seed +
   `resolved_sha256` duoc NHUNG vao config sinh ra, nen config con doc lap
   duong dan va load duoc tu bat ky thu muc nao).
2. Cell 3 sinh danh sach lenh shard theo N x algorithm x seed (moi lenh =
   mot Kaggle job `--train-only`).
3. Cell 4 chay MOT shard do ban chon.
4. Cell 5 aggregate rieng tung N (`--aggregate-only`), roi cell 6 gop bang
   MADDPG vs TD3 paired + Holm theo N.

KHONG goi pipeline monolithic cho tung N.

In [ ]:
import os, sys, subprocess
REPO = "/kaggle/working/STAR_RIS_RSMA_MADDPG"
if not os.path.isdir(REPO):
    REPO = os.path.abspath("..")
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)

In [ ]:
# Sinh config cho tung N bang main.load_config (seed lists + resolved_sha256
# duoc embed -> khong con phu thuoc file seed_split.v1.yaml theo duong dan
# tuong doi). Thay doi duy nhat: num_ris_elements va required_algorithms.
import os, yaml
from main import load_config

base_cfg = load_config("config/config.yaml")   # resolve + verify seed split
N_VALUES = base_cfg["evaluation"]["n_sweep"]
SCAL_ALGOS = ["maddpg", "td3"]
SEEDS = base_cfg["evaluation"]["required_training_seeds"]
cfg_paths = {}
for N in N_VALUES:
    cfg = yaml.safe_load(yaml.safe_dump(base_cfg))   # deep copy DA RESOLVE
    cfg["env"]["num_ris_elements"] = int(N)
    # Ma tran dang ky truoc cua nhanh scalability: chi maddpg vs td3.
    cfg["evaluation"]["required_algorithms"] = SCAL_ALGOS
    out_dir = f"scalability_runs/N{N}"
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"config_N{N}.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    # Config sinh ra phai tu-load duoc (seed split embedded).
    load_config(path)
    cfg_paths[N] = path
    print("OK", path)

In [ ]:
# Danh sach shard N x algorithm x seed (moi dong = mot Kaggle job).
commands = [
    f"python main.py --config {cfg_paths[N]} --train-only "
    f"--algos {algo} --seeds {seed} --run-id scal_N{N}"
    for N in N_VALUES for algo in SCAL_ALGOS for seed in SEEDS
]
for c in commands:
    print(c)
with open("scalability_runs/shard_jobs.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(commands) + "\n")
print(f"\nTong: {len(commands)} shard jobs")

In [ ]:
# Chay MOT shard trong job nay (chon qua bien moi truong).
import subprocess, sys, os
N = int(os.environ.get("SHARD_N", N_VALUES[0]))
SHARD_ALGO = os.environ.get("SHARD_ALGO", SCAL_ALGOS[0])
SHARD_SEED = int(os.environ.get("SHARD_SEED", SEEDS[0]))
cmd = [sys.executable, "main.py", "--config", cfg_paths[N], "--train-only",
       "--algos", SHARD_ALGO, "--seeds", str(SHARD_SEED),
       "--run-id", f"scal_N{N}"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Aggregate rieng tung N sau khi du shard cua N do (2 algos x 8 seeds).
import subprocess, sys
for N in N_VALUES:
    cmd = [sys.executable, "main.py", "--aggregate-only",
           "--load-shards", f"results_revised/shards/scal_N{N}",
           "--run-id", f"scal_N{N}_agg"]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# Gop ket qua theo N tu results_raw.csv cua tung aggregate (KHONG tinh lai),
# paired t-test MADDPG vs TD3 + hieu chinh Holm.
import pandas as pd, numpy as np
from utils import confidence_interval, paired_t_test_p, holm_bonferroni
rows, pvals = [], []
for N in N_VALUES:
    raw = pd.read_csv(f"results_revised/scal_N{N}_agg/tables/results_raw.csv")
    raw = raw[(raw.scenario == "test_bank") & (raw.metric == "sum_rate")]
    seed_means = raw.groupby(["algorithm", "training_seed"]).value.mean().unstack(0)
    m, mci, _ = confidence_interval(seed_means["MADDPG"].values)
    t, tci, _ = confidence_interval(seed_means["TD3"].values)
    p = paired_t_test_p(seed_means["MADDPG"].values, seed_means["TD3"].values)
    pvals.append(p)
    rows.append({"N": N, "MADDPG": m, "MADDPG_CI": mci,
                 "TD3": t, "TD3_CI": tci, "p_paired_t": p})
df = pd.DataFrame(rows)
df["p_holm"] = holm_bonferroni(list(df["p_paired_t"]))
df["significant_5pct"] = df["p_holm"] < 0.05
df.to_csv("scalability_runs/scalability_summary.csv", index=False)
print(df.to_string(index=False))